
# Investigate NRV/RZ Fallback Strategies

This notebook checks:
1) Whether Germany total equals the sum of TSOs.
2) Whether **Betrieblich** (operational) data is available for the Spring 2022 gap.


In [28]:

import io
import os
import requests
import polars as pl
import plotly.express as px

DATA_PATH = "data/processed/all_data.parquet"


In [29]:
# Use NETZTRANSPARENZ_TOKEN or client credentials
NETZTRANSPARENZ_TOKEN = os.getenv('NETZTRANSPARENZ_TOKEN')
CLIENT_ID = os.getenv('NETZTRANSPARENZ_CLIENT_ID')
CLIENT_SECRET = os.getenv('NETZTRANSPARENZ_CLIENT_SECRET')


In [32]:

# Step 1: Summation check (TSO sum vs Germany total)
# Try to use parquet first; if TSO columns are missing, fetch raw RZSaldo for a sample period.

def _load_sample_from_api(start="2023-01-01T00:00Z", end="2023-01-07T00:00Z"):
    if not NETZTRANSPARENZ_TOKEN:
        raise RuntimeError("Set NETZTRANSPARENZ_TOKEN (or bearer_token) before running this cell")
    headers = {"Authorization": NETZTRANSPARENZ_TOKEN if NETZTRANSPARENZ_TOKEN.startswith("Bearer ") else f"Bearer {NETZTRANSPARENZ_TOKEN}"}
    base_url = "https://ds.netztransparenz.de/api/v1/data"
    url = f"{base_url}/NrvSaldo/RZSaldo/Qualitaetsgesichert/{start}/{end}"
    resp = requests.get(url, headers=headers, timeout=60)
    resp.raise_for_status()
    raw = resp.text.lstrip("﻿").lstrip("ï»¿")
    df = pl.read_csv(io.BytesIO(raw.encode("utf-8")), separator=";", infer_schema_length=2000)
    return df

# Try parquet
cols = ["timestamp_utc", "rz_saldo_mw", "50hertz", "amprion", "tennet tso", "transnetbw"]
try:
    df = pl.read_parquet(DATA_PATH)
    missing = [c for c in cols if c not in df.columns]
    if missing:
        print("Missing TSO cols in parquet:", missing)
        df_raw = _load_sample_from_api()
        print("Raw columns:", df_raw.columns)
        display(df_raw.head(3))
    else:
        sample = (
            df.select(cols)
              .filter(pl.col("timestamp_utc").is_between(
                  pl.datetime(2023,1,1, time_zone="UTC"),
                  pl.datetime(2023,2,1, time_zone="UTC"))
              )
              .drop_nulls()
        )
        sample = sample.with_columns(
            (pl.col("50hertz") + pl.col("amprion") + pl.col("tennet tso") + pl.col("transnetbw")).alias("calc_total")
        )
        corr = sample.select(pl.corr("rz_saldo_mw", "calc_total")).item()
        print("Correlation (rz_saldo_mw vs TSO sum):", corr)
        fig = px.scatter(sample.to_pandas(), x="calc_total", y="rz_saldo_mw", title="TSO sum vs Germany total")
        fig.show()
except Exception as exc:
    print("Summation check failed:", exc)


Summation check failed: No such file or directory (os error 2): data/processed/all_data.parquet

This error occurred with the following context stack:
	[1] 'parquet scan'
	[2] 'sink'



In [34]:

# Step 2: Check Betrieblich data availability for gap period (2022-03-01 to 2022-03-05)
base_url = "https://ds.netztransparenz.de/api/v1/data"
start = "2022-03-01T00:00Z"
end = "2022-03-05T00:00Z"

# Token required for API
NETZTRANSPARENZ_TOKEN = os.getenv('NETZTRANSPARENZ_TOKEN')

url = f"{base_url}/NrvSaldo/RZSaldo/Betrieblich/{start}/{end}"
resp = requests.get(url, headers=headers, timeout=60)
print("status", resp.status_code)
resp.raise_for_status()

raw = resp.text.lstrip("﻿").lstrip("ï»¿")
rz_op = pl.read_csv(io.BytesIO(raw.encode("utf-8")), separator=";", infer_schema_length=2000)
print(rz_op.columns)
rz_op.head(3)


status 200
['Datum', 'Zeitzone', 'von', 'bis', 'Datenkategorie', 'Datentyp', 'Einheit', '50Hertz', 'Amprion', 'TenneT TSO', 'TransnetBW']


Datum,Zeitzone,von,bis,Datenkategorie,Datentyp,Einheit,50Hertz,Amprion,TenneT TSO,TransnetBW
str,str,str,str,str,str,str,str,str,str,str
"""01.03.2022""","""UTC""","""00:00""","""00:15""","""RZ-Saldo""","""Betrieblich""","""MW""","""-571,000""","""11,815""","""146,108""","""-25,610"""
"""01.03.2022""","""UTC""","""00:15""","""00:30""","""RZ-Saldo""","""Betrieblich""","""MW""","""-630,000""","""-1,446""","""109,829""","""-47,813"""
"""01.03.2022""","""UTC""","""00:30""","""00:45""","""RZ-Saldo""","""Betrieblich""","""MW""","""-665,000""","""50,206""","""-3,193""","""-63,130"""
